# Modern Coding Practice — Chunked File Transfer (Amazon FAR style)

Closest in spirit to the Morse problem. Parts 1-3 build the core (concurrency at Part 3); Parts 4-5
are stretch tasks (an unreliable channel: drops/reorders/dupes/corruption, then parallel multi-file
verification). Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after
trying.

A chunk is a tuple `(seq, payload, crc)`; a manifest carries `{count, length, checksum}`.

---

## Part 1 — Split & reassemble

`chunk(data: bytes, size) -> list[(seq, payload)]` splits `data` into fixed-size pieces tagged with a
0-based sequence number. `reassemble(chunks) -> bytes` rebuilds the original from chunks given in any
order.

**Lock down:** the last chunk is short; empty input -> no chunks; reassembly must sort by `seq` (the
transport may reorder). Why sequence numbers are mandatory.

In [13]:
def chunk(data, size):
    chunks = []
    num_chunks = (len(data) + size - 1)//size
    for i in range(num_chunks):
        start = i * size
        end = min(len(data), start+size)
        chunks.append((i, data[start:end]))
    return chunks


def reassemble(chunks):
    data = []
    chunks.sort()
    print(chunks)
    for idx, chunk in chunks:
        data.extend(chunk)
    res = bytes(data)
    return res

In [14]:
# --- Part 1 tests ---
def _t1():
    data = b"hello world, this is a test"
    cs = chunk(data, 8)
    assert all(len(p) <= 8 for _, p in cs)
    assert [s for s, _ in cs] == list(range(len(cs)))
    assert reassemble(cs) == data
    assert reassemble(list(reversed(cs))) == data      # order-independent
    assert reassemble(chunk(b"", 8)) == b""
    assert reassemble(chunk(b"abc", 8)) == b"abc"
    print("Part 1 OK")

_t1()

[(0, b'hello wo'), (1, b'rld, thi'), (2, b's is a t'), (3, b'est')]
[(0, b'hello wo'), (1, b'rld, thi'), (2, b's is a t'), (3, b'est')]
[]
[(0, b'abc')]
Part 1 OK


## Part 2 — Checksums & manifest

`pack(data, size) -> (manifest, chunks)` where each chunk is `(seq, payload, crc)` (use
`zlib.crc32`) and `manifest = {count, length, checksum}`. `unpack(manifest, chunks) -> bytes`
verifies every chunk CRC **and** the whole-file checksum, raising `ValueError` on any mismatch.

**Lock down:** per-chunk integrity (detect a bad chunk) vs end-to-end integrity (detect missing /
mis-ordered data). CRC detects corruption, not tampering (that needs a MAC) — say so.

In [23]:
import zlib


def pack(data, size):
    chunks = []
    num_chunks = (len(data) + size - 1)//size
    for i in range(num_chunks):
        start = i * size
        end = min(len(data), start+size)
        d = data[start:end]
        chunks.append((i, d, zlib.crc32(d)))
    manifest = {"count": num_chunks, "length": len(data)}
    return manifest, chunks


def unpack(manifest, chunks):
    data = []
    chunks.sort()
    print(chunks)
    for idx, chunk, chk_sum in chunks:
        if zlib.crc32(chunk) != chk_sum:
            raise ValueError("wrong checksum")
        data.extend(chunk)
    res = bytes(data)
    return res

In [24]:
# --- Part 2 tests ---
def _t2():
    data = b"the quick brown fox jumps"
    manifest, cs = pack(data, 8)
    assert manifest["count"] == len(cs) and manifest["length"] == len(data)
    assert unpack(manifest, cs) == data
    assert unpack(manifest, list(reversed(cs))) == data
    bad = list(cs)
    seq, payload, crc = bad[1]
    bad[1] = (seq, b"X" * len(payload), crc)           # corrupt payload, stale crc
    try:
        unpack(manifest, bad)
    except ValueError:
        pass
    else:
        raise AssertionError("expected CRC mismatch")
    print("Part 2 OK")

_t2()

[(0, b'the quic', 2376367650), (1, b'k brown ', 333166727), (2, b'fox jump', 2921280507), (3, b's', 453955339)]
[(0, b'the quic', 2376367650), (1, b'k brown ', 333166727), (2, b'fox jump', 2921280507), (3, b's', 453955339)]
[(0, b'the quic', 2376367650), (1, b'XXXXXXXX', 333166727), (2, b'fox jump', 2921280507), (3, b's', 453955339)]
Part 2 OK


## Part 3 — Concurrent send / receive

`transfer(data, size) -> bytes`: a sender thread packs `data` and pushes chunks onto a bounded
`queue.Queue`; a receiver thread drains until a sentinel, then `unpack`s and returns the bytes.

**Discuss:** sentinel shutdown, bounded queue = backpressure when the sender outruns the receiver,
clean thread join; this is I/O/coordination (threads fine) — Part 5 does CPU-bound verification with
processes.

In [ ]:
import queue
import threading


def transfer(data, size):
    raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    data = b"concurrent chunked transfer works " * 3
    assert transfer(data, 8) == data
    assert transfer(b"", 8) == b""
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Unreliable channel

Networks drop, reorder, duplicate, and corrupt. Build `ReliableReceiver(manifest)`:

- `receive(chunk)`: accept `(seq, payload, crc)`. Discard CRC-failures (count `corrupt_dropped`),
  ignore an already-seen `seq` (count `duplicates`), otherwise store it.
- `missing() -> list[int]`: sequence numbers not yet received (drives retransmit requests).
- `assemble() -> bytes`: raise `ValueError` if anything is missing; else verify the whole-file
  checksum and return the bytes.

**Discuss:** why CRC-before-dedup ordering matters, idempotency under duplicates, sliding-window /
selective-ACK retransmit, and bounding receiver memory.

In [ ]:
import zlib


class ReliableReceiver:
    def __init__(self, manifest):
        raise NotImplementedError

    def receive(self, chunk):
        raise NotImplementedError

    def missing(self):
        raise NotImplementedError

    def assemble(self):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    import random
    data = b"unreliable channels need reordering and dedup and crc checks!"
    manifest, chunks = pack(data, 8)

    rng = random.Random(0)
    stream = []
    for c in chunks:
        stream.append(c)                                   # the clean chunk
        if rng.random() < 0.5:
            stream.append(c)                               # a duplicate
        seq, payload, crc = c
        stream.append((seq, b"~" + payload[1:], crc))      # a corrupted copy (bad crc)
    rng.shuffle(stream)

    rx = ReliableReceiver(manifest)
    for c in stream:
        rx.receive(c)
    assert rx.missing() == []
    assert rx.assemble() == data
    assert rx.corrupt_dropped > 0 and rx.duplicates > 0

    rx2 = ReliableReceiver(manifest)
    for c in chunks[1:]:                                    # drop seq 0 entirely
        rx2.receive(c)
    assert 0 in rx2.missing()
    try:
        rx2.assemble()
    except ValueError:
        pass
    else:
        raise AssertionError("expected missing-chunk error")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel multi-file verify

`parallel_verify(files) -> dict[name, bool]`: verify many received files at once (CRC per chunk +
whole-file checksum), CPU-bound, across processes with `ProcessPoolExecutor`. `files` is
`dict[name, (manifest, chunks)]`; the worker is `transfer_workers.verify`.

**Discuss:** GIL (hashing is CPU-bound -> processes), pickling cost of shipping payloads to workers
(send paths/offsets instead at real scale), and that verification is per-file independent.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import transfer_workers


def parallel_verify(files) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    man_g, cs_g = pack(b"a perfectly good file payload here", 8)
    man_b, cs_b = pack(b"this one will be corrupted in transit", 8)
    seq, payload, crc = cs_b[0]
    cs_b = [(seq, b"~" + payload[1:], crc)] + cs_b[1:]      # corrupt the first chunk
    files = {"good": (man_g, cs_g), "bad": (man_b, cs_b)}
    assert parallel_verify(files) == {"good": True, "bad": False}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Retransmit protocol: ACK/NACK, windows, timeouts, exponential backoff.
- Streaming reassembly with bounded memory; flushing in-order prefixes early.
- Stronger integrity (SHA-256 / HMAC), compression, resumable transfers.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import zlib
import queue
import threading
from concurrent.futures import ProcessPoolExecutor
import transfer_workers


def ref_chunk(data, size):
    return [(i // size, data[i:i + size]) for i in range(0, len(data), size)]


def ref_reassemble(chunks):
    return b"".join(p for _, p in sorted(chunks))


def ref_pack(data, size):
    chunks = [(i // size, data[i:i + size], zlib.crc32(data[i:i + size]))
              for i in range(0, len(data), size)]
    manifest = {"count": len(chunks), "length": len(data), "checksum": zlib.crc32(data)}
    return manifest, chunks


def ref_unpack(manifest, chunks):
    for seq, payload, crc in chunks:
        if zlib.crc32(payload) != crc:
            raise ValueError("chunk %d corrupt" % seq)
    data = b"".join(p for _, p, _ in sorted(chunks))
    if len(data) != manifest["length"] or zlib.crc32(data) != manifest["checksum"]:
        raise ValueError("whole-file checksum mismatch")
    return data


def ref_transfer(data, size):
    manifest, chunks = ref_pack(data, size)
    q = queue.Queue(maxsize=16)
    sentinel = object()
    result = {}

    def sender():
        for c in chunks:
            q.put(c)
        q.put(sentinel)

    def receiver():
        got = []
        while True:
            item = q.get()
            if item is sentinel:
                break
            got.append(item)
        result["data"] = ref_unpack(manifest, got)

    r = threading.Thread(target=receiver)
    s = threading.Thread(target=sender)
    r.start(); s.start(); s.join(); r.join()
    return result["data"]


class RefReliableReceiver:
    def __init__(self, manifest):
        self.manifest = manifest
        self.have = {}                          # seq -> payload
        self.corrupt_dropped = 0
        self.duplicates = 0

    def receive(self, chunk):
        seq, payload, crc = chunk
        if zlib.crc32(payload) != crc:          # check integrity first
            self.corrupt_dropped += 1
            return
        if seq in self.have:
            self.duplicates += 1
            return
        self.have[seq] = payload

    def missing(self):
        return [s for s in range(self.manifest["count"]) if s not in self.have]

    def assemble(self):
        miss = self.missing()
        if miss:
            raise ValueError("missing chunks: %r" % miss)
        data = b"".join(self.have[s] for s in range(self.manifest["count"]))
        if len(data) != self.manifest["length"] or zlib.crc32(data) != self.manifest["checksum"]:
            raise ValueError("whole-file checksum mismatch")
        return data


def ref_parallel_verify(files):
    items = [(name, man, chunks) for name, (man, chunks) in files.items()]
    with ProcessPoolExecutor() as ex:
        return dict(ex.map(transfer_workers.verify, items))


_m, _c = ref_pack(b"reference check payload", 8)
assert ref_unpack(_m, list(reversed(_c))) == b"reference check payload"
assert ref_transfer(b"abcdefghijklmnop", 5) == b"abcdefghijklmnop"
_rx = RefReliableReceiver(_m)
for _ch in reversed(_c):
    _rx.receive(_ch)
assert _rx.assemble() == b"reference check payload"
assert ref_parallel_verify({"g": ref_pack(b"good", 8)}) == {"g": True}
print("reference solutions OK")